# Fence Detection Demo

This notebook demonstrates the agentic fence detection approach on PDF drawings.

In [ ]:
import sys
sys.path.insert(0, '..')

import fitz
import math
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.collections import LineCollection
import numpy as np
from PIL import Image
from io import BytesIO

# Import our fence detection modules
from fence_detector_agentic import (
    extract_all_lines,
    cluster_by_proximity_and_orientation,
    filter_fence_like_clusters,
    analyze_page_structure,
    agentic_fence_detection,
    find_lines_near_indicator  # New indicator-driven function
)

%matplotlib inline
plt.rcParams['figure.figsize'] = [16, 10]
plt.rcParams['figure.dpi'] = 100

print("Imports loaded successfully!")

## 1. Load PDF and Select Page

In [ ]:
# Load PDF
PDF_PATH = "../gold_standard/subset_gold/selected_pages_no_annotations.pdf"
doc = fitz.open(PDF_PATH)
print(f"Loaded PDF with {len(doc)} pages")

# Select page (0-indexed)
PAGE_NUM = 2  # Change this to analyze different pages
page = doc[PAGE_NUM]
print(f"\nPage {PAGE_NUM + 1}: {page.rect.width:.0f} x {page.rect.height:.0f} pts")

In [ ]:
# Render page as image
def render_page(page, dpi=100):
    pix = page.get_pixmap(dpi=dpi)
    img = Image.open(BytesIO(pix.tobytes("png")))
    return np.array(img)

page_img = render_page(page, dpi=100)
plt.figure(figsize=(16, 10))
plt.imshow(page_img)
plt.title(f"Page {PAGE_NUM + 1} - Original")
plt.axis('off')
plt.show()

## 2. Extract Vector Lines

In [ ]:
# Extract all lines with minimum length filter
MIN_LENGTH = 15.0  # pts
lines = extract_all_lines(page, min_length=MIN_LENGTH)

print(f"Extracted {len(lines)} lines (min length: {MIN_LENGTH} pts)")

# Show length distribution
lengths = [l['length_pts'] for l in lines]
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.hist(lengths, bins=50, edgecolor='black')
plt.xlabel('Line Length (pts)')
plt.ylabel('Count')
plt.title('Line Length Distribution')

# Show angle distribution
angles = [l['angle'] for l in lines]
plt.subplot(1, 2, 2)
plt.hist(angles, bins=36, edgecolor='black')
plt.xlabel('Angle (degrees)')
plt.ylabel('Count')
plt.title('Line Angle Distribution')
plt.tight_layout()
plt.show()

In [ ]:
# Visualize all extracted lines
def plot_lines_on_page(page_img, lines, page, title="Lines", color='blue', alpha=0.5):
    fig, ax = plt.subplots(figsize=(16, 10))
    
    # Show page image
    h, w = page_img.shape[:2]
    ax.imshow(page_img, extent=[0, page.rect.width, page.rect.height, 0])
    
    # Plot lines
    line_segments = [[(l['start'][0], l['start'][1]), (l['end'][0], l['end'][1])] for l in lines]
    lc = LineCollection(line_segments, colors=color, linewidths=1, alpha=alpha)
    ax.add_collection(lc)
    
    ax.set_xlim(0, page.rect.width)
    ax.set_ylim(page.rect.height, 0)
    ax.set_title(f"{title} ({len(lines)} lines)")
    plt.show()

plot_lines_on_page(page_img, lines, page, "All Extracted Lines", color='cyan', alpha=0.3)

## 3. Cluster Lines by Proximity & Orientation

In [ ]:
# Cluster lines
DISTANCE_THRESHOLD = 25.0  # pts
ANGLE_THRESHOLD = 20.0  # degrees

clusters = cluster_by_proximity_and_orientation(
    lines,
    distance_threshold=DISTANCE_THRESHOLD,
    angle_threshold=ANGLE_THRESHOLD
)

print(f"Formed {len(clusters)} clusters")

# Show cluster size distribution
cluster_sizes = sorted([len(c.lines) for c in clusters], reverse=True)
print(f"Top 10 cluster sizes: {cluster_sizes[:10]}")

plt.figure(figsize=(12, 4))
plt.bar(range(min(30, len(cluster_sizes))), cluster_sizes[:30])
plt.xlabel('Cluster Rank')
plt.ylabel('Number of Lines')
plt.title('Cluster Size Distribution (Top 30)')
plt.show()

In [ ]:
# Visualize clusters with different colors
def plot_clusters(page_img, clusters, page, title="Clusters", min_lines=5):
    fig, ax = plt.subplots(figsize=(16, 10))
    ax.imshow(page_img, extent=[0, page.rect.width, page.rect.height, 0])
    
    # Generate colors
    cmap = plt.cm.get_cmap('tab20')
    
    # Filter and plot clusters
    large_clusters = [c for c in clusters if len(c.lines) >= min_lines]
    large_clusters.sort(key=lambda c: len(c.lines), reverse=True)
    
    for i, cluster in enumerate(large_clusters[:20]):  # Top 20
        color = cmap(i % 20)
        line_segments = [[(l['start'][0], l['start'][1]), (l['end'][0], l['end'][1])] 
                        for l in cluster.lines]
        lc = LineCollection(line_segments, colors=[color], linewidths=2, alpha=0.8)
        ax.add_collection(lc)
        
        # Draw bbox
        x0, y0, x1, y1 = cluster.bbox
        rect = patches.Rectangle((x0, y0), x1-x0, y1-y0, 
                                  linewidth=1, edgecolor=color, facecolor='none', linestyle='--')
        ax.add_patch(rect)
    
    ax.set_xlim(0, page.rect.width)
    ax.set_ylim(page.rect.height, 0)
    ax.set_title(f"{title} ({len(large_clusters)} clusters with >= {min_lines} lines)")
    plt.show()

plot_clusters(page_img, clusters, page, "Line Clusters", min_lines=5)

## 4. Filter Fence-Like Clusters

In [ ]:
# Filter for fence-like characteristics
fence_clusters = filter_fence_like_clusters(
    clusters,
    min_lines=5,
    min_length_pts=200.0,
    min_aspect_ratio=4.0
)

print(f"Found {len(fence_clusters)} fence-like clusters")
print("\nCluster details:")
for i, c in enumerate(fence_clusters[:10]):
    length_ft = (c.total_length_pts / 72.0) * 30 / 12.0  # Assuming 1"=30' scale
    print(f"  {i+1}. {len(c.lines):4d} lines, {c.total_length_pts:7.0f} pts ({length_ft:6.1f} ft), {c.orientation}")

In [ ]:
# Visualize fence-like clusters
def plot_fence_clusters(page_img, fence_clusters, page, scale_factor=30.0):
    fig, ax = plt.subplots(figsize=(16, 10))
    ax.imshow(page_img, extent=[0, page.rect.width, page.rect.height, 0])
    
    colors = {'horizontal': 'red', 'vertical': 'blue', 'diagonal': 'green'}
    
    for i, cluster in enumerate(fence_clusters):
        color = colors.get(cluster.orientation, 'purple')
        
        # Draw lines
        line_segments = [[(l['start'][0], l['start'][1]), (l['end'][0], l['end'][1])] 
                        for l in cluster.lines]
        lc = LineCollection(line_segments, colors=[color], linewidths=2, alpha=0.8)
        ax.add_collection(lc)
        
        # Draw bbox with label
        x0, y0, x1, y1 = cluster.bbox
        rect = patches.Rectangle((x0, y0), x1-x0, y1-y0, 
                                  linewidth=2, edgecolor=color, facecolor=color, alpha=0.1)
        ax.add_patch(rect)
        
        # Label with length
        length_ft = (cluster.total_length_pts / 72.0) * scale_factor / 12.0
        ax.text(x0, y0-5, f"{length_ft:.0f} ft", fontsize=8, color=color, fontweight='bold')
    
    ax.set_xlim(0, page.rect.width)
    ax.set_ylim(page.rect.height, 0)
    ax.set_title(f"Fence-Like Clusters ({len(fence_clusters)} regions)\nRed=Horizontal, Blue=Vertical, Green=Diagonal")
    
    # Add legend
    total_ft = sum((c.total_length_pts / 72.0) * scale_factor / 12.0 for c in fence_clusters)
    ax.text(0.02, 0.02, f"Total: {total_ft:.0f} ft", transform=ax.transAxes, 
            fontsize=14, fontweight='bold', color='white',
            bbox=dict(boxstyle='round', facecolor='black', alpha=0.8))
    
    plt.show()

plot_fence_clusters(page_img, fence_clusters, page, scale_factor=30.0)

## 5. Full Agentic Detection Pipeline

In [ ]:
# Run full INDICATOR-DRIVEN pipeline
SCALE_FACTOR = 30.0  # 1" = 30' (adjust based on your drawing)

# NEW: Indicator-driven detection with tunable parameters
result = agentic_fence_detection(
    page, 
    scale_factor=SCALE_FACTOR, 
    verbose=True,
    search_radius=150.0,       # How far from indicator to search for seed lines
    connection_tolerance=25.0   # How close endpoints must be to connect lines
)

print(f"\n{'='*60}")
print(f"Indicator-driven: {len(result.get('indicator_regions', []))} regions")
print(f"Fallback detected: {len(result.get('unmatched_regions', []))} regions")

In [ ]:
# Display results summary
print("\n" + "="*60)
print("DETECTION RESULTS")
print("="*60)
print(f"\nTotal Fence Regions: {len(result['fence_regions'])}")
print(f"Total Length: {result['total_length_feet']:.1f} feet")
print(f"Scale Factor: 1\" = {result['scale_factor']}'")

print("\nRegion Details:")
for i, region in enumerate(result['fence_regions']):
    indicator = region.get('indicator', 'N/A')
    print(f"  {i+1}. {region['orientation']:10s} | {region['segment_count']:4d} segments | {region['length_feet']:7.1f} ft | {indicator}")

### Visualize Agentic Detection Results

This shows the detected fence regions overlaid on the PDF page with bounding boxes and measurements.

In [ ]:
# Visualize the INDICATOR-DRIVEN agentic detection results
def plot_agentic_results(page_img, result, page, scale_factor=30.0):
    """Visualize indicator-driven fence detection results."""
    fig, ax = plt.subplots(figsize=(18, 12))
    ax.imshow(page_img, extent=[0, page.rect.width, page.rect.height, 0])
    
    # Different colors for indicator-driven vs fallback
    for i, region in enumerate(result['fence_regions']):
        region_type = region.get('type', 'unknown')
        
        # Color by type: indicator-driven = green, fallback = orange
        if region_type == 'indicator_driven':
            color = '#00FF00'  # Bright green for indicator-driven
            label_prefix = "✓"
        else:
            color = '#FFA500'  # Orange for fallback
            label_prefix = "?"
        
        # Get bbox and draw rectangle
        bbox = region.get('bbox', (0, 0, 0, 0))
        x0, y0, x1, y1 = bbox
        
        # Draw filled rectangle
        rect = patches.Rectangle((x0, y0), x1-x0, y1-y0, 
                                  linewidth=3, edgecolor=color, 
                                  facecolor=color, alpha=0.15)
        ax.add_patch(rect)
        
        # Draw border
        rect_border = patches.Rectangle((x0, y0), x1-x0, y1-y0, 
                                         linewidth=2, edgecolor=color, 
                                         facecolor='none')
        ax.add_patch(rect_border)
        
        # Draw indicator marker if present
        if region.get('indicator_bbox'):
            ix0, iy0, ix1, iy1 = region['indicator_bbox']
            ind_rect = patches.Rectangle((ix0-5, iy0-5), ix1-ix0+10, iy1-iy0+10,
                                          linewidth=3, edgecolor='#FF0000',
                                          facecolor='yellow', alpha=0.5)
            ax.add_patch(ind_rect)
        
        # Label
        length_ft = region.get('length_feet', 0)
        indicator = region.get('indicator', None)
        if indicator:
            label = f"{label_prefix} {indicator}\\n{length_ft:.0f} ft"
        else:
            label = f"{label_prefix} {region.get('orientation', '')}\\n{length_ft:.0f} ft"
        
        ax.text(x0 + 5, y0 + 20, label, fontsize=9, color='white', fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.3', facecolor=color, alpha=0.9))
    
    ax.set_xlim(0, page.rect.width)
    ax.set_ylim(page.rect.height, 0)
    
    # Title
    n_indicator = len(result.get('indicator_regions', []))
    n_fallback = len(result.get('unmatched_regions', []))
    title = f"INDICATOR-DRIVEN Fence Detection\\n"
    title += f"{n_indicator} from indicators (green) | {n_fallback} fallback (orange) | Total: {result['total_length_feet']:.0f} ft"
    ax.set_title(title, fontsize=14, fontweight='bold')
    
    # Legend
    from matplotlib.lines import Line2D
    legend_elements = [
        Line2D([0], [0], color='#00FF00', linewidth=6, label=f'Indicator-driven ({n_indicator})'),
        Line2D([0], [0], color='#FFA500', linewidth=6, label=f'Fallback detected ({n_fallback})'),
        patches.Patch(facecolor='yellow', edgecolor='red', label='Text indicator location'),
    ]
    ax.legend(handles=legend_elements, loc='upper right', fontsize=10)
    
    plt.tight_layout()
    plt.show()
    
    # Summary table
    print("\\n" + "="*80)
    print("INDICATOR-DRIVEN DETECTION RESULTS")
    print("="*80)
    print(f"{'#':<4} {'Type':<18} {'Indicator':<25} {'Segments':<10} {'Length (ft)'}")
    print("-"*80)
    for i, region in enumerate(result['fence_regions']):
        rtype = region.get('type', 'unknown')[:16]
        ind = (region.get('indicator') or region.get('orientation', '-'))[:23]
        print(f"{i+1:<4} {rtype:<18} {ind:<25} {region['segment_count']:<10} {region['length_feet']:.1f}")
    print("-"*80)
    print(f"{'TOTAL':<4} {'':<18} {'':<25} {sum(r['segment_count'] for r in result['fence_regions']):<10} {result['total_length_feet']:.1f}")

# Visualize!
plot_agentic_results(page_img, result, page, scale_factor=SCALE_FACTOR)

## 6. Parameter Tuning

Adjust these parameters to improve detection for your specific drawings:

In [ ]:
# Interactive parameter testing
def test_parameters(page, min_length, distance_thresh, angle_thresh, min_cluster_lines, min_aspect):
    """Test different parameter combinations."""
    lines = extract_all_lines(page, min_length=min_length)
    clusters = cluster_by_proximity_and_orientation(
        lines,
        distance_threshold=distance_thresh,
        angle_threshold=angle_thresh
    )
    fence_clusters = filter_fence_like_clusters(
        clusters,
        min_lines=min_cluster_lines,
        min_length_pts=200.0,
        min_aspect_ratio=min_aspect
    )
    
    total_ft = sum((c.total_length_pts / 72.0) * 30 / 12.0 for c in fence_clusters)
    
    print(f"Parameters: min_len={min_length}, dist={distance_thresh}, angle={angle_thresh}, min_lines={min_cluster_lines}, aspect={min_aspect}")
    print(f"  → {len(lines)} lines, {len(clusters)} clusters, {len(fence_clusters)} fence regions, {total_ft:.0f} ft")
    
    return fence_clusters

# Test different combinations
print("Testing different parameters:\n")
test_parameters(page, 10, 20, 15, 5, 3)
test_parameters(page, 15, 25, 20, 5, 4)
test_parameters(page, 20, 30, 25, 3, 5)
test_parameters(page, 10, 35, 30, 3, 3)

## 8. Method Comparison

We compare 6 different fence detection methods against ground truth annotations:

| Method | Description |
|--------|-------------|
| **Global Clustering** | Baseline: cluster all lines by proximity/orientation, filter by shape |
| **Aggressive Clustering** | Relaxed parameters for more inclusive grouping |
| **Conservative Clustering** | Strict parameters for fewer false positives |
| **Indicator-Driven** | Find fence keywords in text, then measure nearby lines |
| **Dimension Lines** | Find numeric text (measurements), match to nearby lines |
| **Oracle Best-Match** | Upper bound: use GT locations to find best matching lines |

In [ ]:
# Import comparison methods
from fence_detection_comparison import (
    parse_ground_truth,
    method_global_clustering,
    method_aggressive_clustering,
    method_conservative_clustering,
    method_indicator_driven,
    method_dimension_lines,
    method_oracle_best_match,
    evaluate_method
)

# Load ground truth annotations
GT_CSV = "../gold_standard/subset_gold/df_annotations_sub.csv"
gt_by_page = parse_ground_truth(GT_CSV)

print("Ground Truth Summary:")
print("-" * 50)
for page_num, annotations in sorted(gt_by_page.items()):
    lengths = [a.length_ft for a in annotations if a.length_ft]
    total = sum(lengths)
    print(f"  Page {page_num}: {len(annotations)} annotations, {len(lengths)} with measurements, total={total:.0f} ft")

In [ ]:
# Run all methods on current page
TEST_PAGE = PAGE_NUM + 1  # Convert to 1-indexed for GT lookup
SCALE = 30.0

# Re-open PDF for testing
doc = fitz.open(PDF_PATH)
test_page = doc[PAGE_NUM]
gt = gt_by_page.get(TEST_PAGE, [])

print(f"Testing on Page {TEST_PAGE}")
print(f"Ground Truth: {len([g for g in gt if g.length_ft])} measurements, {sum(g.length_ft for g in gt if g.length_ft):.0f} ft total")
print("=" * 70)

# Run each method and collect results
all_results = {}

methods = [
    ("Global Clustering", lambda p: method_global_clustering(p, SCALE)),
    ("Aggressive Clustering", lambda p: method_aggressive_clustering(p, SCALE)),
    ("Conservative Clustering", lambda p: method_conservative_clustering(p, SCALE)),
    ("Indicator-Driven", lambda p: method_indicator_driven(p, SCALE)),
    ("Dimension Lines", lambda p: method_dimension_lines(p, SCALE)),
    ("Oracle Best-Match", lambda p: method_oracle_best_match(p, gt, SCALE)),
]

for name, method_fn in methods:
    result = method_fn(test_page)
    result.method_name = name
    metrics = evaluate_method(result, gt, verbose=False)
    all_results[name] = {'result': result, 'metrics': metrics}
    print(f"\n{name}:")
    print(f"  Detected: {result.total_length_ft:.0f} ft ({len(result.regions)} regions)")
    print(f"  Accuracy: {metrics['length_accuracy']:.1%} | Error: {metrics['length_error_pct']:.0f}% | Recall: {metrics['recall']:.1%}")

In [ ]:
# Visualize results for each method
import matplotlib.patches as patches

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

for idx, (name, data) in enumerate(all_results.items()):
    ax = axes[idx]
    result = data['result']
    metrics = data['metrics']
    
    # Render page as image
    pix = test_page.get_pixmap(matrix=fitz.Matrix(0.5, 0.5))
    img = np.frombuffer(pix.samples, dtype=np.uint8).reshape(pix.height, pix.width, pix.n)
    ax.imshow(img)
    
    # Draw detected regions as bounding boxes
    for region in result.regions:
        bbox = region.get('bbox', (0, 0, 0, 0))
        x0, y0, x1, y1 = bbox
        # Scale coordinates for 0.5x rendering
        rect = patches.Rectangle((x0*0.5, y0*0.5), (x1-x0)*0.5, (y1-y0)*0.5,
                                  linewidth=2, edgecolor='red', facecolor='red', alpha=0.3)
        ax.add_patch(rect)
        
        # Label with length
        length_ft = region.get('length_ft', 0)
        ax.text(x0*0.5, y0*0.5-5, f"{length_ft:.0f}ft", fontsize=7, color='red', fontweight='bold')
    
    # Title with metrics
    ax.set_title(f"{name}\\nDet: {result.total_length_ft:.0f}ft | Acc: {metrics['length_accuracy']:.0%} | Err: {metrics['length_error_pct']:.0f}%", fontsize=10)
    ax.axis('off')

plt.tight_layout()
plt.suptitle(f"Method Comparison - Page {TEST_PAGE}", fontsize=14, y=1.02)
plt.show()

---
## SymPointV2 Semantic Segmentation

This section runs the SymPointV2 model to perform semantic segmentation on the PDF's vector elements. The model classifies each path element (lines, arcs, etc.) into semantic categories including fence-related classes.

**Key steps:**
1. Load the SymPointV2 model checkpoint
2. Convert PDF page to SVG and extract path data
3. Run inference to get per-element semantic predictions
4. Visualize results with fence-related elements highlighted

## 9. SymPoint2 Neural Network Inference

Run the SymPointV2 model on the PDF page to get semantic segmentation predictions for vector elements. This shows which lines are classified as fences, walls, doors, etc.

In [ ]:
# Convert PDF page to SVG and run inference
from svgpathtools import parse_path
import xml.etree.ElementTree as ET
import re

def pdf_page_to_svg_data(page):
    """Convert PDF page to SVG and extract path data for SymPointV2."""
    # Export page as SVG
    svg_output = page.get_svg_image()
    
    # Handle both bytes and string return types (PyMuPDF version dependent)
    if isinstance(svg_output, bytes):
        content = svg_output.decode('utf-8', errors='ignore')
    else:
        content = svg_output
    
    # Clean up invalid characters
    content = re.sub(r'&#x[0-9a-fA-F]*;', '', content)
    content = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f]', '', content)
    
    root = ET.fromstring(content)
    
    # Get dimensions
    viewbox = root.attrib.get('viewBox', '0 0 1000 1000')
    parts = viewbox.split()
    width = float(parts[2]) if len(parts) >= 3 else 1000
    height = float(parts[3]) if len(parts) >= 4 else 1000
    
    args = []
    commands = []
    lengths = []
    path_elements = []  # Store for visualization
    
    COMMANDS = ['Line', 'Arc', 'circle', 'ellipse']
    
    for elem in root.iter():
        if elem.tag.endswith('path') and 'd' in elem.attrib:
            d = elem.attrib['d']
            if not d.strip():
                continue
            try:
                path_repre = parse_path(d)
                if len(path_repre) == 0:
                    continue
                
                path_type = path_repre[0].__class__.__name__
                if path_type not in COMMANDS:
                    path_type = 'Line'
                commands.append(COMMANDS.index(path_type))
                
                length = path_repre.length()
                lengths.append(length)
                
                # Get path points
                points = []
                for seg in path_repre:
                    if hasattr(seg, 'start') and hasattr(seg, 'end'):
                        points.extend([seg.start.real, seg.start.imag,
                                      seg.end.real, seg.end.imag])
                
                while len(points) < 8:
                    points.extend(points[-2:] if len(points) >= 2 else [0, 0])
                args.extend(points[:8])
                
                # Store for visualization
                path_elements.append({
                    'start': (points[0], points[1]),
                    'end': (points[2], points[3]),
                    'length': length
                })
                
            except Exception as e:
                continue
    
    if len(args) == 0:
        return None, []
    
    return {
        'width': width,
        'height': height,
        'args': args,
        'commands': commands,
        'lengths': lengths
    }, path_elements

# Convert test page
svg_data, path_elements = pdf_page_to_svg_data(test_page)
print(f"Extracted {len(path_elements)} path elements from page")

In [ ]:
# Convert PDF page to SVG and run inference
from svgpathtools import parse_path
import xml.etree.ElementTree as ET
import re

def pdf_page_to_svg_data(page):
    """Convert PDF page to SVG and extract path data for SymPointV2."""
    # Export page as SVG
    svg_bytes = page.get_svg_image()
    
    # Parse SVG
    content = svg_bytes.decode('utf-8', errors='ignore')
    content = re.sub(r'&#x[0-9a-fA-F]*;', '', content)
    content = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f]', '', content)
    
    root = ET.fromstring(content)
    
    # Get dimensions
    viewbox = root.attrib.get('viewBox', '0 0 1000 1000')
    parts = viewbox.split()
    width = float(parts[2]) if len(parts) >= 3 else 1000
    height = float(parts[3]) if len(parts) >= 4 else 1000
    
    args = []
    commands = []
    lengths = []
    path_elements = []  # Store for visualization
    
    COMMANDS = ['Line', 'Arc', 'circle', 'ellipse']
    
    for elem in root.iter():
        if elem.tag.endswith('path') and 'd' in elem.attrib:
            d = elem.attrib['d']
            if not d.strip():
                continue
            try:
                path_repre = parse_path(d)
                if len(path_repre) == 0:
                    continue
                
                path_type = path_repre[0].__class__.__name__
                if path_type not in COMMANDS:
                    path_type = 'Line'
                commands.append(COMMANDS.index(path_type))
                
                length = path_repre.length()
                lengths.append(length)
                
                # Get path points
                points = []
                for seg in path_repre:
                    if hasattr(seg, 'start') and hasattr(seg, 'end'):
                        points.extend([seg.start.real, seg.start.imag,
                                      seg.end.real, seg.end.imag])
                
                while len(points) < 8:
                    points.extend(points[-2:] if len(points) >= 2 else [0, 0])
                args.extend(points[:8])
                
                # Store for visualization
                path_elements.append({
                    'start': (points[0], points[1]),
                    'end': (points[2], points[3]),
                    'length': length
                })
                
            except Exception as e:
                continue
    
    if len(args) == 0:
        return None, None
    
    return {
        'width': width,
        'height': height,
        'args': args,
        'commands': commands,
        'lengths': lengths
    }, path_elements

# Convert test page
svg_data, path_elements = pdf_page_to_svg_data(test_page)
print(f"Extracted {len(path_elements)} path elements from page")

In [ ]:
# Prepare batch and run SymPointV2 inference
def prepare_sympoint_batch(data, min_points=2048):
    """Prepare data for SymPointV2 model inference."""
    width, height = data['width'], data['height']
    coords = np.array(data['args']).reshape(-1, 8)
    
    # Normalize coordinates
    coords[:, 0::2] = coords[:, 0::2] / width
    coords[:, 1::2] = coords[:, 1::2] / height
    
    num = coords.shape[0]
    max_num = max(num, min_points)
    
    # Create coordinate tensor (x, y, z) - center of each path element
    coord = np.zeros((max_num, 3))
    coord_x = np.mean(coords[:, 0::2], axis=1)
    coord_y = np.mean(coords[:, 1::2], axis=1)
    coord[:num, 0] = coord_x
    coord[:num, 1] = coord_y
    
    # Create feature tensor (7 dims): arcs, lens, ctype(4), widths
    feat = np.zeros((max_num, 7))
    
    # Arcs (angle with horizontal)
    dx = coords[:, 2] - coords[:, 0]
    dy = coords[:, 3] - coords[:, 1]
    arcs = np.arctan2(dy, dx) / np.pi
    feat[:num, 0] = arcs
    
    # Lengths (normalized)
    max_lengths = max(width, height)
    lengths_arr = np.array(data['lengths'])
    lens = np.clip(lengths_arr, 0, max_lengths) / max_lengths
    feat[:num, 1] = lens
    
    # Command type one-hot (4 classes)
    commands_arr = np.array(data['commands'])
    ctype = np.eye(4)[np.clip(commands_arr, 0, 3)]
    feat[:num, 2:6] = ctype
    
    # Widths
    feat[:num, 6] = 1.0
    
    # Semantic labels (background for inference)
    LABEL_NUM = len(CLASS_NAMES)
    semantic_labels = np.ones((max_num, 2), dtype=np.int64) * -1
    semantic_labels[:num, 0] = LABEL_NUM
    
    # Create batch tensors
    coords_t = torch.from_numpy(coord).float()
    feats_t = torch.from_numpy(feat).float()
    labels_t = torch.from_numpy(semantic_labels).long()
    offsets_t = torch.tensor([max_num], dtype=torch.int32)
    lengths_t = torch.from_numpy(lengths_arr).float()
    lengths_t = torch.nn.functional.pad(lengths_t, (0, max_num - num))
    layer_ids_t = torch.zeros(max_num, dtype=torch.long)
    
    return (coords_t, feats_t, labels_t, offsets_t, lengths_t, layer_ids_t), num

# Prepare and run inference
if svg_data:
    batch, num_points = prepare_sympoint_batch(svg_data)
    
    with torch.no_grad():
        result = model(batch, return_loss=False)
    
    # Get semantic predictions
    sem_scores = result['semantic_scores']
    sem_preds = torch.argmax(sem_scores, dim=1).cpu().numpy()[:num_points]
    
    # Count predictions per class
    unique, counts = np.unique(sem_preds, return_counts=True)
    
    print(f"SymPointV2 Predictions ({num_points} elements):")
    print("-" * 40)
    for cls_id, count in sorted(zip(unique, counts), key=lambda x: -x[1]):
        if cls_id < len(CLASS_NAMES):
            pct = 100 * count / num_points
            print(f"  {CLASS_NAMES[cls_id]:20s}: {count:5d} ({pct:5.1f}%)")
else:
    print("No SVG data extracted from page")

In [ ]:
# Visualize SymPointV2 semantic segmentation results
def visualize_sympoint_results(page, path_elements, predictions, class_names, figsize=(16, 12)):
    """Visualize SymPointV2 predictions overlaid on the PDF page."""
    # Render page to image
    pix = page.get_pixmap(dpi=100)
    img = np.frombuffer(pix.samples, dtype=np.uint8).reshape(pix.height, pix.width, pix.n)
    if pix.n == 4:
        img = img[:, :, :3]
    
    # Scale factors
    scale_x = pix.width / page.rect.width
    scale_y = pix.height / page.rect.height
    
    # Create colormap for classes
    np.random.seed(42)
    num_classes = len(class_names)
    colors = plt.cm.tab20(np.linspace(0, 1, num_classes))
    
    fig, axes = plt.subplots(1, 2, figsize=figsize)
    
    # Left: Full page with all predictions
    axes[0].imshow(img)
    axes[0].set_title('SymPointV2 Semantic Predictions', fontsize=14)
    
    # Overlay predictions
    for i, (elem, pred) in enumerate(zip(path_elements, predictions)):
        if pred < num_classes:
            color = colors[pred]
            start = (elem['start'][0] * scale_x, elem['start'][1] * scale_y)
            end = (elem['end'][0] * scale_x, elem['end'][1] * scale_y)
            axes[0].plot([start[0], end[0]], [start[1], end[1]], 
                        color=color, linewidth=1.5, alpha=0.7)
    
    axes[0].axis('off')
    
    # Right: Fence-related classes only
    axes[1].imshow(img)
    axes[1].set_title('Fence-Related Predictions Only', fontsize=14)
    
    # Find fence-related class indices
    fence_keywords = ['fence', 'wall', 'gate', 'barrier', 'rail', 'guard']
    fence_class_ids = []
    for i, name in enumerate(class_names):
        name_lower = name.lower()
        if any(kw in name_lower for kw in fence_keywords):
            fence_class_ids.append(i)
    
    fence_count = 0
    for i, (elem, pred) in enumerate(zip(path_elements, predictions)):
        if pred in fence_class_ids:
            fence_count += 1
            color = colors[pred]
            start = (elem['start'][0] * scale_x, elem['start'][1] * scale_y)
            end = (elem['end'][0] * scale_x, elem['end'][1] * scale_y)
            axes[1].plot([start[0], end[0]], [start[1], end[1]], 
                        color=color, linewidth=2, alpha=0.9)
    
    axes[1].axis('off')
    
    # Add legend
    legend_elements = []
    for cls_id in fence_class_ids:
        count = np.sum(predictions == cls_id)
        if count > 0:
            legend_elements.append(plt.Line2D([0], [0], color=colors[cls_id], 
                                              linewidth=2, label=f'{class_names[cls_id]} ({count})'))
    
    if legend_elements:
        axes[1].legend(handles=legend_elements, loc='upper right', fontsize=10)
    
    plt.tight_layout()
    plt.suptitle(f'SymPointV2 Results: {fence_count} fence-related elements detected', 
                 fontsize=16, y=1.02)
    plt.show()
    
    return fence_count

# Visualize
if svg_data and 'sem_preds' in dir():
    fence_count = visualize_sympoint_results(test_page, path_elements, sem_preds, CLASS_NAMES)
    print(f"\nTotal fence-related elements: {fence_count}")
else:
    print("No predictions available for visualization")

---
## Combined Results Summary

The notebook demonstrates multiple approaches to fence detection:

| Method | Approach | Best For |
|--------|----------|----------|
| **Global Clustering** | K-means on all lines | PDFs without layers |
| **Indicator-Driven** | Lines near fence indicators | Labeled drawings |
| **Dimension Lines** | Text-matched measurements | Annotated plans |
| **Aggressive Clustering** | Larger search radius | Sparse drawings |
| **Conservative Clustering** | Stricter proximity | Dense drawings |
| **Oracle Best-Match** | Ground truth matching | Evaluation only |
| **SymPointV2** | Deep learning segmentation | Complex drawings |

Each method has tradeoffs between precision (avoiding false positives) and recall (finding all fences).

## Method Performance Summary

| Method | Description | Detected (ft) | Accuracy | Error % | Recall |
|--------|-------------|---------------|----------|---------|--------|
| **Global Clustering** | Baseline: clusters all lines by proximity | High | Low | High | - |
| **Aggressive Clustering** | Relaxed clustering params (larger gap tolerance) | High | Low | High | - |
| **Conservative Clustering** | Strict params (tighter gaps, min length) | Moderate | Moderate | Moderate | - |
| **Indicator-Driven** | Finds fence keywords in text, clusters nearby lines | Low | Low | High | Low |
| **Dimension Lines** | Extracts dimension text + finds lines with matching lengths | Moderate | Moderate | Moderate | Moderate |
| **Oracle Best-Match** | Uses GT locations to find best-matching lines (upper bound) | Varies | ~89% | ~13% | High |

### Key Findings

1. **Oracle Best-Match** achieves ~89% accuracy, indicating that single-line matching can find most fence segments but some measurements span multiple connected segments.

2. **Clustering methods** tend to over-detect because they group all nearby lines, including non-fence elements (borders, grids, etc.).

3. **Indicator-Driven** under-detects because fence-related keywords are sparse in the PDF text layer.

4. **Dimension Lines** shows promise but requires reliable text extraction of measurement values.

### Recommendations

- Combine dimension line detection with indicator-driven filtering
- Implement connected-line grouping to handle multi-segment fences
- Add layer-based filtering if PDF has named layers (e.g., "FENCE" layer)

## 7. Cleanup

In [ ]:
doc.close()
print("Done!")